# Notebook v4 — memperluas eksperimen terkontrol ke CSE-CIC-IDS2018

**Mengapa ini perlu.** Seluruh korelasi antar kategori di naskah bertumpu pada enam kategori dari satu
dataset. Pada n = 6, hanya |ρ| ≳ 0,83 yang signifikan, dan dua prediktor utama — *detection headroom*
dan *synthetic intrusion rate* — berkorelasi satu sama lain pada ρ = −0,94 sehingga tidak bisa
dipisahkan. Keduanya adalah keberatan terkuat yang akan diajukan reviewer, dan keduanya hanya bisa
dijawab dengan menambah jumlah kategori.

Notebook ini menjalankan **eksperimen terkontrol yang sama persis** pada CSE-CIC-IDS2018: setiap
kategori dipangkas ke 29 sampel latih, dinaikkan ke target penyeimbangan yang sama, dengan seed dan
fold yang sama. Hasilnya enam kategori tambahan, sehingga analisis gabungan memakai **n = 12**.

| | n | |ρ| minimum agar signifikan (α = 0,05) |
|---|---|---|
| sekarang | 6 | 0,83 |
| setelah notebook ini | 12 | 0,58 |

**Yang disengaja berbeda dari v3.** Keluarga model dibatasi pada Random Forest dan XGBoost; MLP
dilewati karena naskah sendiri menyatakannya sebagai kaki terlemah. Metode dibatasi pada base dan
SMOTE; dekomposisi ROS sudah tegak pada data 2017.

**Ruang fitur tidak disatukan.** CIC-IDS2017 dan CSE-CIC-IDS2018 punya himpunan fitur yang berbeda,
dan memaksakan irisan keduanya akan mengubah geometri yang justru sedang diukur. Setiap dataset
dianalisis di ruang fiturnya sendiri; yang digabungkan adalah **peringkat kategori**, dan Spearman
memang bekerja pada peringkat. Bagian S melaporkan hasil per dataset dan gabungan berdampingan.

| Bagian | Isi | Perkiraan waktu |
|---|---|---|
| 0–2 | Setup, unduh, bersihkan, cache | ±60–90 menit sekali saja |
| P | Eksperimen terkontrol, dipotong per sesi | ±3 jam per potongan |
| Q | Geometri subsampel dan tujuan titik sintetis | ±20 menit |
| R | Tujuan false negative | < 2 menit |
| S | Analisis gabungan 2017 + 2018 | < 2 menit |

Folder Drive yang dipakai sama dengan v2 dan v3, sehingga bagian S bisa membaca hasil 2017 langsung.


## 0. Pengaturan

In [ ]:
# ====== PENGATURAN ======
SEEDS           = [42, 43, 44]       # identik dengan 2017
N_FOLDS         = 5
CAP             = 30000              # batas record per kategori, identik dengan 2017
TARGET          = 24000              # target penyeimbangan, identik dengan 2017
N_CONTROL       = 29                 # titik kontrol, identik dengan 2017
RF_TREES        = 100
XGB_TREES       = 300
XGB_DEPTH       = 6
XGB_LR          = 0.1
RANDOM_STATE    = 42

MODELS          = ["rf", "xgb"]      # MLP sengaja dilewati
METHODS         = ["base", "smote"]  # ROS sudah tegak pada data 2017

# --- potongan yang dilatih pada sesi ini ---
RUN_SEEDS       = [42]               # lalu [43], lalu [44]
RUN_MODELS      = ["rf", "xgb"]
RUN_METHODS     = ["base", "smote"]
TIME_BUDGET_MIN = 170                # berhenti rapi sebelum Colab memutus

RUN_P_CONTROL   = True               # eksperimen terkontrol
RUN_Q_GEOMETRY  = True               # geometri + tujuan titik sintetis
N_SYNTH         = 4000
USE_CACHE       = True

import os, time, json, math, zipfile, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTDIR = "/content/drive/MyDrive/juti_rerun"
except Exception:
    OUTDIR = "./juti_rerun"
os.makedirs(OUTDIR, exist_ok=True)
CSE_DIR = os.path.join(OUTDIR, "cse2018_full"); os.makedirs(CSE_DIR, exist_ok=True)
print("Folder hasil:", OUTDIR)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-learn==1.6.1", "imbalanced-learn==0.14.2", "xgboost", "boto3"], check=False)
import sklearn, imblearn, xgboost
print("scikit-learn", sklearn.__version__, "| imbalanced-learn", imblearn.__version__,
      "| xgboost", xgboost.__version__)

## 1. Unduh dan bersihkan CSE-CIC-IDS2018

Dataset lengkapnya sekitar 6,4 GB dan 16 juta record — terlalu besar untuk dimuat sekaligus di Colab.
Sel ini membacanya **per potongan** dan mengambil sampel acak seragam sebanyak `CAP` record per
kategori memakai *reservoir sampling*, sehingga memori yang dipakai tetap terbatas pada ukuran
sampel akhir, bukan ukuran berkas.

Pemetaan label mengikuti taksonomi yang dipakai pada CIC-IDS2017, agar kategori kedua dataset dapat
diperbandingkan. Label apa pun yang tidak dikenali akan **dicetak dan dihentikan**, bukan dibuang
diam-diam — kalau itu terjadi, kirimkan daftarnya kepada saya.

Hasil bersihnya disimpan ke `cse2018_clean.npz` sehingga sesi berikutnya tidak perlu mengunduh ulang.


In [ ]:
CLEAN18 = os.path.join(OUTDIR, "cse2018_clean.npz")

def category_of_2018(label):
    """Petakan label CSE-CIC-IDS2018 ke taksonomi yang dipakai pada CIC-IDS2017."""
    l = " ".join(str(label).strip().split()).lower()
    if l in ("benign", "normal"):              return "BENIGN"
    if l == "bot":                             return "Botnet"
    if l.startswith("ddos") or "ddos" in l:    return "DDoS"
    if l.startswith("dos"):                    return "DoS"
    if "infilt" in l:                          return "Infiltration"
    if "sql injection" in l:                   return "Web Attack"
    if "brute force -web" in l or "brute force -xss" in l: return "Web Attack"
    if "bruteforce" in l or "brute force" in l: return "Brute Force"
    return None

CATS18 = ["BENIGN", "Botnet", "DDoS", "DoS", "Web Attack", "Brute Force", "Infiltration"]
ATTACK18 = [c for c in CATS18 if c != "BENIGN"]

def fetch_cse2018_all():
    have = sorted(glob.glob(os.path.join(CSE_DIR, "*.csv")))
    if have:
        print(f"{len(have)} berkas sudah ada di {CSE_DIR}")
        return have
    import boto3
    from botocore import UNSIGNED
    from botocore.config import Config
    s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED), region_name="ca-central-1")
    prefix = "Processed Traffic Data for ML Algorithms/"
    keys = [o["Key"] for o in s3.list_objects_v2(Bucket="cse-cic-ids2018",
                                                 Prefix=prefix).get("Contents", [])]
    want = sorted(k for k in keys if k.endswith(".csv"))
    print(f"Mengunduh {len(want)} berkas (sekitar 6,4 GB, sekali saja):")
    for k in want:
        dst = os.path.join(CSE_DIR, os.path.basename(k))
        if not os.path.exists(dst):
            t0 = time.time()
            s3.download_file("cse-cic-ids2018", k, dst)
            print(f"  {os.path.basename(k)}  {os.path.getsize(dst)/1e6:.0f} MB  "
                  f"({time.time()-t0:.0f} s)")
    return sorted(glob.glob(os.path.join(CSE_DIR, "*.csv")))


DROP_COLS = ["Flow ID", "Src IP", "Dst IP", "Timestamp", "Src Port"]

def common_columns(files):
    """Berkas CSE-CIC-IDS2018 tidak seragam: satu hari memuat kolom identitas tambahan."""
    sets = []
    for f in files:
        c = [x.strip() for x in pd.read_csv(f, nrows=0).columns]
        sets.append(set(c))
    inter = set.intersection(*sets)
    extra = sorted(set.union(*sets) - inter)
    if extra:
        print("Kolom yang tidak ada di semua berkas, dibuang:", extra)
    cols = [c for c in sorted(inter) if c not in DROP_COLS]
    return cols


def load_capped(files, cols, cap=CAP, chunk=400_000, seed=RANDOM_STATE):
    """Sampel seragam cap record per kategori memakai kunci acak.

    Setiap baris diberi kunci U(0,1); yang disimpan adalah cap baris dengan kunci
    terkecil. Hasilnya persis sama dengan reservoir sampling, tetapi seluruhnya
    tervektorisasi sehingga sanggup menangani 16 juta record."""
    rng = np.random.default_rng(seed)
    keep = {c: None for c in CATS18}        # (DataFrame, kunci)
    seen = {c: 0 for c in CATS18}
    unknown = {}
    feat = [c for c in cols if c != "Label"]
    for f in files:
        for d in pd.read_csv(f, low_memory=False, chunksize=chunk,
                             usecols=lambda x: x.strip() in cols):
            d.columns = [c.strip() for c in d.columns]
            d = d[d["Label"].astype(str).str.strip() != "Label"]
            if not len(d):
                continue
            cat = d["Label"].map(category_of_2018)
            bad = d.loc[cat.isna(), "Label"].astype(str).str.strip()
            for bname, bn in bad.value_counts().items():
                unknown[bname] = unknown.get(bname, 0) + int(bn)
            d = d.loc[cat.notna()]
            cats = cat[cat.notna()].values
            for c in np.unique(cats):
                g = d.loc[cats == c, feat]
                seen[c] += len(g)
                k = rng.random(len(g))
                if keep[c] is None:
                    gg, kk = g, k
                else:
                    gg = pd.concat([keep[c][0], g], ignore_index=True)
                    kk = np.r_[keep[c][1], k]
                if len(gg) > cap:
                    sel = np.argpartition(kk, cap)[:cap]
                    gg, kk = gg.iloc[sel].reset_index(drop=True), kk[sel]
                keep[c] = (gg, kk)
        print("  selesai " + os.path.basename(f) + "  " +
              " ".join(f"{c}={seen[c]:,}" for c in CATS18 if seen[c]))
    if unknown:
        raise RuntimeError("Label tidak dikenali (kirimkan daftar ini): " + json.dumps(unknown))
    out, labs = [], []
    for c in CATS18:
        if keep[c] is None:
            print(f"  PERINGATAN: kategori {c} tidak ditemukan")
            continue
        out.append(keep[c][0]); labs += [c] * len(keep[c][0])
    return pd.concat(out, ignore_index=True), np.array(labs, dtype=object).astype(str), seen


if USE_CACHE and os.path.exists(CLEAN18):
    z = np.load(CLEAN18, allow_pickle=True)
    X18, y18 = z["X"], z["y"].astype(str)
    FEAT18 = list(z["features"])
    print(f"Cache dibaca: {X18.shape[0]:,} record x {X18.shape[1]} fitur")
    print(pd.Series(y18).value_counts().to_string())
else:
    files = fetch_cse2018_all()
    if not files:
        raise RuntimeError("Tidak ada berkas CSV. Unggah manual ke " + CSE_DIR)
    COLS18 = common_columns(files)
    Xdf, y18, seen = load_capped(files, COLS18)
    print("\nJumlah mentah per kategori:", {c: f"{n:,}" for c, n in seen.items() if n})
    for c_ in Xdf.select_dtypes(include="object").columns:       # Timestamp dan sejenisnya
        Xdf[c_] = pd.to_numeric(Xdf[c_], errors="coerce")
    Xdf = Xdf.loc[:, Xdf.notna().any()]
    Xdf = Xdf.replace([np.inf, -np.inf], np.nan)
    keep = Xdf.notna().all(axis=1)
    Xdf, y18 = Xdf[keep], y18[keep.values]
    const = [c_ for c_ in Xdf.columns if Xdf[c_].nunique() <= 1]
    Xdf = Xdf.drop(columns=const)
    seen_h, dup = {}, []
    for c_ in Xdf.columns:
        h = pd.util.hash_pandas_object(Xdf[c_], index=False).values.sum()
        if h in seen_h and Xdf[c_].equals(Xdf[seen_h[h]]):
            dup.append(c_)
        else:
            seen_h.setdefault(h, c_)
    Xdf = Xdf.drop(columns=dup)
    FEAT18 = Xdf.columns.tolist()
    X18 = Xdf.values.astype(np.float64)
    print(f"\nKolom konstan dibuang: {len(const)} | kolom kembar: {len(dup)}")
    print(f"Fitur akhir: {len(FEAT18)} | record: {len(X18):,}")
    print(pd.Series(y18).value_counts().to_string())
    np.savez_compressed(CLEAN18, X=X18, y=np.asarray(y18, dtype=object).astype(str),
                        features=np.array(FEAT18, dtype=object))
    print("\nDisimpan ke cache:", CLEAN18)

CLASSES18 = sorted(np.unique(y18))
TARGET_CATS18 = [c for c in ATTACK18 if (y18 == c).sum() >= 60]
print("\nKategori yang ikut eksperimen terkontrol:", TARGET_CATS18)

## P. Eksperimen terkontrol pada CSE-CIC-IDS2018

Desainnya identik dengan bagian C di v2 dan bagian G di v3, hanya datasetnya yang berganti: setiap
kategori serangan dipangkas ke 29 sampel latih di dalam setiap fold, lalu dinaikkan ke target 24.000
bersama kategori lain yang berada di bawah target. Benih subsampel memakai rumus yang sama,
`seed * 100 + fold`, sehingga bisa direproduksi.

Saklar `RUN_SEEDS`, `RUN_MODELS`, `RUN_METHODS` dan `TIME_BUDGET_MIN` bekerja persis seperti di v3.
Urutan yang disarankan: satu seed per sesi, sekitar tiga jam masing-masing.


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

CLS2ID18 = {c: i for i, c in enumerate(CLASSES18)}
ID2CLS18 = np.array(CLASSES18)

def make_model(name, seed):
    if name == "rf":
        return RandomForestClassifier(n_estimators=RF_TREES, n_jobs=-1, random_state=seed)
    if name == "xgb":
        return XGBClassifier(n_estimators=XGB_TREES, max_depth=XGB_DEPTH, learning_rate=XGB_LR,
                             subsample=0.9, colsample_bytree=0.9, tree_method="hist",
                             objective="multi:softmax", num_class=len(CLASSES18),
                             n_jobs=-1, random_state=seed, verbosity=0)
    raise ValueError(name)

def fit_predict(name, seed, Xs, ys, Xte):
    m = make_model(name, seed)
    if name == "xgb":
        m.fit(Xs, np.array([CLS2ID18[v] for v in ys], dtype=int))
        return ID2CLS18[np.asarray(m.predict(Xte)).astype(int)]
    m.fit(Xs, ys)
    return np.asarray(m.predict(Xte)).astype(str)

def smote_strategy(y):
    vc = pd.Series(y).value_counts()
    return {c: TARGET for c, n in vc.items() if n < TARGET}

def counts_for(y_true, y_pred, c):
    return (int(((y_pred == c) & (y_true == c)).sum()),
            int(((y_pred == c) & (y_true != c)).sum()),
            int(((y_pred != c) & (y_true == c)).sum()))

CKP = os.path.join(OUTDIR, "secP_controlled_cse2018.csv")
COLS_P = ["seed", "fold", "category", "model", "method", "n_train",
          "tp", "fp", "fn", "fn_dest", "fit_seconds"]
doneP = pd.read_csv(CKP) if os.path.exists(CKP) else pd.DataFrame(columns=COLS_P)
keysP = set(zip(doneP.seed, doneP.fold, doneP.category, doneP.model, doneP.method))

SLICE = [(s, f, c, mo, me) for s in RUN_SEEDS for f in range(N_FOLDS)
         for c in TARGET_CATS18 for mo in RUN_MODELS for me in RUN_METHODS]
todo = [k for k in SLICE if k not in keysP]
full = len(SEEDS) * N_FOLDS * len(TARGET_CATS18) * len(MODELS) * len(METHODS)
print(f"Matriks lengkap : {full} model | sudah selesai: {len(keysP)}")
print(f"Potongan sesi ini: {len(todo)} dari {len(SLICE)} model")

t_start, n_run, stopped = time.time(), 0, False
for seed in (RUN_SEEDS if RUN_P_CONTROL else []):
    if stopped: break
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, te) in enumerate(skf.split(X18, y18)):
        if stopped: break
        if not any((seed, fold, c, mo, me) not in keysP
                   for c in TARGET_CATS18 for mo in RUN_MODELS for me in RUN_METHODS):
            continue
        sc = StandardScaler().fit(X18[tr])
        Xtr_f, Xte_f = sc.transform(X18[tr]), sc.transform(X18[te])
        ytr_f, yte_f = y18[tr], y18[te]
        for c in TARGET_CATS18:
            if stopped: break
            rng_t = np.random.default_rng(seed * 100 + fold)    # rumus benih identik dengan 2017
            idx_c = np.where(ytr_f == c)[0]
            if len(idx_c) > N_CONTROL:
                idx_c = rng_t.choice(idx_c, size=N_CONTROL, replace=False)
            keep_idx = np.r_[np.where(ytr_f != c)[0], idx_c]
            Xk, yk = Xtr_f[keep_idx], ytr_f[keep_idx]
            for me in RUN_METHODS:
                if stopped: break
                todo_m = [mo for mo in RUN_MODELS if (seed, fold, c, mo, me) not in keysP]
                if not todo_m:
                    continue
                Xs, ys = ((SMOTE(sampling_strategy=smote_strategy(yk), k_neighbors=5,
                                 random_state=seed).fit_resample(Xk, yk))
                          if me == "smote" else (Xk, yk))
                for mo in todo_m:
                    if TIME_BUDGET_MIN and (time.time() - t_start) / 60 >= TIME_BUDGET_MIN:
                        stopped = True; break
                    t0 = time.time()
                    pred = fit_predict(mo, seed, Xs, ys, Xte_f)
                    dt = time.time() - t0
                    tp, fp, fn = counts_for(yte_f, pred, c)
                    dest = pd.Series(pred[(yte_f == c) & (pred != c)]).value_counts().to_dict()
                    doneP = pd.concat([doneP, pd.DataFrame([dict(
                        seed=seed, fold=fold, category=c, model=mo, method=me,
                        n_train=len(idx_c), tp=tp, fp=fp, fn=fn,
                        fn_dest=json.dumps({str(k): int(v) for k, v in dest.items()}),
                        fit_seconds=round(dt, 1))])], ignore_index=True)
                    doneP.to_csv(CKP, index=False)
                    keysP.add((seed, fold, c, mo, me)); n_run += 1
                    left = len([k for k in SLICE if k not in keysP])
                    avg = (time.time() - t_start) / n_run
                    print(f"seed {seed} fold {fold} {c:13s} {mo:3s} {me:5s} {dt:5.0f} s "
                          f"| sisa {left} ≈ {left*avg/60:.0f} menit")

left = len([k for k in SLICE if k not in keysP])
if stopped:
    print(f"\nBerhenti karena batas waktu. Sisa potongan ini: {left} model. "
          f"Jalankan ulang dengan RUN_* yang sama untuk melanjutkan.")
elif left == 0:
    print(f"\nPotongan selesai. Lanjutkan dengan RUN_SEEDS berikutnya, "
          f"atau jalankan bagian Q bila semua seed sudah selesai.")

## Q. Geometri subsampel dan tujuan titik sintetis

Mengulang pengukuran bagian B/J pada data 2018: tipologi Napierała–Stefanowski, dispersion ratio δ,
synthetic intrusion rate ι, dan kelas tempat setiap titik sintetis yang menyusup mendarat. Tanpa
bagian ini, kategori 2018 tidak bisa masuk ke analisis korelasi gabungan.


In [ ]:
K_NN = 5

def example_types(X_min, X_other, k=K_NN):
    Xb = np.vstack([X_min, X_other])
    is_min = np.r_[np.ones(len(X_min), bool), np.zeros(len(X_other), bool)]
    _, ind = NearestNeighbors(n_neighbors=k + 1).fit(Xb).kneighbors(X_min)
    same = is_min[ind[:, 1:]].sum(axis=1)
    return {"safe (%)": np.mean(same >= 4) * 100,
            "borderline (%)": np.mean((same == 2) | (same == 3)) * 100,
            "rare (%)": np.mean(same == 1) * 100,
            "outlier (%)": np.mean(same == 0) * 100}

def dispersion_ratio(X_min, nn_other):
    d_in, _ = NearestNeighbors(n_neighbors=2).fit(X_min).kneighbors(X_min)
    d_out, _ = nn_other.kneighbors(X_min, n_neighbors=1)
    return d_in[:, 1].mean() / max(d_out[:, 0].mean(), 1e-12)

def synth_and_destination(X_min, X_oth, y_oth, rng, k=K_NN, n_new=N_SYNTH):
    k = min(k, len(X_min) - 1)
    _, ind = NearestNeighbors(n_neighbors=k + 1).fit(X_min).kneighbors(X_min)
    i = rng.integers(0, len(X_min), n_new)
    j = ind[i, rng.integers(1, k + 1, n_new)]
    lam = rng.random((n_new, 1))
    X_new = X_min[i] + lam * (X_min[j] - X_min[i])
    d_min, _ = NearestNeighbors(n_neighbors=1).fit(X_min).kneighbors(X_new)
    nn_o = NearestNeighbors(n_neighbors=1).fit(X_oth)
    d_oth, idx_o = nn_o.kneighbors(X_new)
    intr = d_oth[:, 0] < d_min[:, 0]
    return intr, y_oth[idx_o[:, 0]]

CKQ = os.path.join(OUTDIR, "secQ_shape_cse2018.csv")
CKQ2 = os.path.join(OUTDIR, "secQ_synth_dest_cse2018.csv")
doneQ = pd.read_csv(CKQ) if os.path.exists(CKQ) else pd.DataFrame(
    columns=["seed", "fold", "category"])
keysQ = set(zip(doneQ.seed, doneQ.fold, doneQ.category)) if len(doneQ) else set()

for seed in (SEEDS if RUN_Q_GEOMETRY else []):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, te) in enumerate(skf.split(X18, y18)):
        if all((seed, fold, c) in keysQ for c in TARGET_CATS18):
            continue
        sc = StandardScaler().fit(X18[tr])
        Xtr_f, ytr_f = sc.transform(X18[tr]), y18[tr]
        for c in TARGET_CATS18:
            if (seed, fold, c) in keysQ:
                continue
            rng_t = np.random.default_rng(seed * 100 + fold)
            idx_c = np.where(ytr_f == c)[0]
            if len(idx_c) > N_CONTROL:
                idx_c = rng_t.choice(idx_c, size=N_CONTROL, replace=False)
            X_min, X_oth = Xtr_f[idx_c], Xtr_f[ytr_f != c]
            nn_o = NearestNeighbors(n_neighbors=1).fit(X_oth)
            intr, _ = synth_and_destination(X_min, X_oth, ytr_f[ytr_f != c],
                                            np.random.default_rng(seed + fold), n_new=2000)
            row = dict(seed=seed, fold=fold, category=c, **example_types(X_min, X_oth))
            row["dispersion ratio"] = dispersion_ratio(X_min, nn_o)
            row["synthetic intrusion (%)"] = 100 * intr.mean()
            doneQ = pd.concat([doneQ, pd.DataFrame([row])], ignore_index=True)
            doneQ.to_csv(CKQ, index=False); keysQ.add((seed, fold, c))
        print(f"seed {seed} fold {fold} geometri selesai")

# tujuan titik sintetis, sekali per kategori pada split 80:20 tetap
if RUN_Q_GEOMETRY:
    from sklearn.model_selection import train_test_split
    Xtr_raw, _, ytr_g, _ = train_test_split(X18, y18, test_size=0.2, stratify=y18,
                                            random_state=RANDOM_STATE)
    scg = StandardScaler().fit(Xtr_raw); Xtr_g = scg.transform(Xtr_raw)
    rng_j = np.random.default_rng(RANDOM_STATE)
    rows = []
    for c in TARGET_CATS18:
        X_cat = Xtr_g[ytr_g == c]
        pick = rng_j.choice(len(X_cat), size=min(N_CONTROL, len(X_cat)), replace=False)
        X_min = X_cat[pick]
        mask = ytr_g != c
        intr, dest = synth_and_destination(X_min, Xtr_g[mask], ytr_g[mask], rng_j)
        d = pd.Series(dest[intr])
        share = (d.value_counts(normalize=True) * 100) if len(d) else pd.Series(dtype=float)
        other = share.drop(labels=["BENIGN"], errors="ignore")
        rows.append(dict(Category=c, n_synth=len(intr),
                         **{"synthetic intrusion (%)": 100 * intr.mean(),
                            "to BENIGN (%)": float(share.get("BENIGN", 0.0)) if len(share) else np.nan,
                            "top non-BENIGN": str(other.idxmax()) if len(other) else ""}))
        print(f"{c:13s} ι={100*intr.mean():5.1f}%  ke BENIGN={rows[-1]['to BENIGN (%)']:5.1f}%")
    pd.DataFrame(rows).to_csv(CKQ2, index=False)
    print("\n", pd.DataFrame(rows).round(1).to_string(index=False))

## S. Analisis gabungan 2017 + 2018

Inilah alasan seluruh notebook ini ada. Sel di bawah membaca hasil terkontrol kedua dataset,
menghitung baseline F1, ΔF1 dan R_c per kategori, lalu melaporkan korelasi Spearman **per dataset dan
gabungan berdampingan**.

Tiga hal yang perlu diperhatikan saat membaca keluarannya:

* Pada n = 12, ambang signifikansi turun dari |ρ| ≈ 0,83 ke sekitar 0,58.
* Kolinearitas antara baseline F1 dan ι dihitung ulang pada data gabungan. Kalau ia turun dari −0,94,
  kedua prediktor mulai bisa dipisahkan dan klaim naskah dapat dipertajam. Kalau tetap tinggi, itu
  temuan tersendiri: keduanya memang mengukur hal yang sama.
* Kategori dengan nama sama di kedua dataset (Botnet, DoS, DDoS, Web Attack, Infiltration) adalah
  jenis serangan yang sama pada jaringan berbeda, jadi 12 titik itu tidak sepenuhnya saling bebas.
  Sel ini juga mencetak korelasi dengan satu dataset ditahan, sebagai pemeriksaan.


In [ ]:
from scipy.stats import spearmanr

def f1_from(tp, fp, fn):
    return 200.0 * tp / max(2 * tp + fp + fn, 1)

def summarise(path, tag):
    if not os.path.exists(path):
        print(f"  {path} tidak ada — dilewati"); return pd.DataFrame()
    d = pd.read_csv(path)
    d = d[d.method.isin(["base", "smote"]) & d.model.isin(["rf", "xgb"])]
    rows = []
    for (c, mo), sub in d.groupby(["category", "model"]):
        b = sub[sub.method == "base"].set_index(["seed", "fold"]).sort_index()
        s = sub[sub.method == "smote"].set_index(["seed", "fold"]).sort_index()
        common = b.index.intersection(s.index)
        if len(common) < 6:
            continue
        b, s = b.loc[common], s.loc[common]
        f1b = np.array([f1_from(*r) for r in b[["tp", "fp", "fn"]].values])
        f1s = np.array([f1_from(*r) for r in s[["tp", "fp", "fn"]].values])
        fnb, fns = b["fn"].sum(), s["fn"].sum()
        rows.append(dict(dataset=tag, category=c, model=mo, n_pairs=len(common),
                         base_F1=f1b.mean(), delta=(f1s - f1b).mean(),
                         R_c=100.0 * (fnb - fns) / max(fnb, 1)))
    return pd.DataFrame(rows)

a = summarise(os.path.join(OUTDIR, "secG_controlled_v3.csv"), "CIC-IDS2017")
b = summarise(os.path.join(OUTDIR, "secP_controlled_cse2018.csv"), "CSE-CIC-IDS2018")
res = pd.concat([a, b], ignore_index=True)
if not len(res):
    raise SystemExit("Belum ada hasil untuk dirangkum.")

sh17 = os.path.join(OUTDIR, "secC_subsample_shape.csv")
sh18 = os.path.join(OUTDIR, "secQ_shape_cse2018.csv")
geo = []
for p, tag in [(sh17, "CIC-IDS2017"), (sh18, "CSE-CIC-IDS2018")]:
    if os.path.exists(p):
        g = pd.read_csv(p).groupby("category")[["dispersion ratio",
                                                "synthetic intrusion (%)"]].mean().reset_index()
        g["dataset"] = tag; geo.append(g)
geo = pd.concat(geo, ignore_index=True) if geo else pd.DataFrame()
m = res.merge(geo, on=["dataset", "category"], how="left") if len(geo) else res

print("=== Ringkasan per kategori ===")
print(m.round(2).to_string(index=False), "\n")

def rep(x, y, lab):
    ok = pd.notna(x) & pd.notna(y)
    if ok.sum() < 4:
        return f"  {lab:34s} n terlalu kecil"
    r, p = spearmanr(x[ok], y[ok])
    star = "*" if p < 0.05 else " "
    return f"  {lab:34s} rho = {r:+.2f} (p = {p:.3f}) n = {ok.sum()}{star}"

for mo in sorted(res.model.unique()):
    print(f"=== model {mo.upper()} ===")
    for tag in ["CIC-IDS2017", "CSE-CIC-IDS2018", "GABUNGAN"]:
        s = m[m.model == mo] if tag == "GABUNGAN" else m[(m.model == mo) & (m.dataset == tag)]
        if len(s) < 4:
            print(f"  {tag}: n = {len(s)}, dilewati"); continue
        print(f"  -- {tag} (n = {len(s)}) --")
        print(rep(100 - s.base_F1, s.R_c, "headroom vs R_c"))
        print(rep(s.base_F1, s.delta, "baseline F1 vs dF1"))
        if "synthetic intrusion (%)" in s:
            print(rep(s["synthetic intrusion (%)"], s.R_c, "iota vs R_c"))
            print(rep(s["dispersion ratio"], s.R_c, "dispersi vs R_c"))
            print(rep(s.base_F1, s["synthetic intrusion (%)"], "KOLINEARITAS baseline vs iota"))
    print()

print("=== Pemeriksaan ketergantungan: satu dataset ditahan ===")
for mo in sorted(res.model.unique()):
    for hold in ["CIC-IDS2017", "CSE-CIC-IDS2018"]:
        s = m[(m.model == mo) & (m.dataset != hold)]
        if len(s) >= 4 and "synthetic intrusion (%)" in s:
            print(f"  {mo.upper()} tanpa {hold}:"
                  f" {rep(100 - s.base_F1, s.R_c, 'headroom vs R_c').strip()}")
m.to_csv(os.path.join(OUTDIR, "v4_pooled_summary.csv"), index=False)
print("\nRingkasan disimpan: v4_pooled_summary.csv")